# L36 · 窗函数（window function）原理 — 矩形窗的旁瓣（sidelobe）泄漏，Hann / Hamming / Blackman 对比

**今日目标**：先比较 Rectangular / Hann / Hamming / Blackman 的形状，再看它们如何改变信号边缘；最后阅读 `io.py` 与 `windows.py`，把概念对应到 Aurora 源码。

**目标**：比较 Rectangular / Hann / Hamming / Blackman 四种窗函数（window function）的时域形状与频率响应，理解矩形窗旁瓣（sidelobe）泄漏的来源，以及平滑窗如何压制泄漏。

**为什么对 Aurora 重要**：STFT（短时傅里叶变换）的每一帧都乘以 Hann 窗；`src/aurora/audio/windows.py` 就是这节课的直接实现，选错窗函数会导致频谱泄漏污染梅尔特征。

## 本课剧情：给声音边缘做修整

截取一段信号时，边缘处信号值突然跳到零，FFT 会把这个跳变解读为高频成分，形成旁瓣污染频谱。窗函数是一串与信号等长的权重，逐点相乘后把两端压向零，旁瓣随之减小。今天先直接观察矩形窗与 Hann 窗的旁瓣差异，再阅读 `aurora.audio.windows` 的源码，看这三行数学如何落成代码。

## 1. 先看窗函数本身

窗函数是一串和信号等长的权重。把信号逐点乘上这些权重，片段边缘会被压低，中间部分保留得更多。

- Rectangular：所有位置都是 1，相当于不处理边缘。
- Hann / Blackman：两端接近 0，边缘最柔和。
- Hamming：两端不为 0，保留一点边缘能量。

用四个数字就能描述一个窗的边界行为：首端值 `w[0]`、尾端值 `w[-1]`、峰值 `w.max()`、峰值位置 `argmax(w)`。Hann / Blackman 首尾接近 0，把边缘压至几乎为零；Hamming 首尾约 0.08，保留少量边缘能量；Rectangular 首尾为 1，完全不做压制。边界值越接近 0，频谱旁瓣越低，但主瓣（main lobe）越宽——旁瓣抑制与频谱分辨率之间不可避免的权衡。

## 实验入口：把声音拆成可观察的数组

这里用64个采样点、采样率（sample rate，sr）200 Hz的短正弦，让 `signal * window` 的结果可以直接打印，便于确认边缘值是否被压低。

In [ ]:
import numpy as np
from aurora.audio import hann, hamming, blackman
print('就绪')

## 动手观察：序列怎样一步步变成信号

改 `sample_rate` 或 `duration`，观察采样点数 `N = int(sample_rate * duration)` 如何变化；下一格的窗权重数组长度将与 `N` 保持一致。

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## 代码实验：比较几种窗函数的边缘

打印 `w[:3]` 和 `w[-3:]`，直接看矩形窗边缘值为 1.0 而 Hann 窗边缘接近 0 这一差异。

In [ ]:
import numpy as np

N = 16
windows = {
    'rectangular': np.ones(N),
    'hann': np.hanning(N),
    'hamming': np.hamming(N),
}

for name, w in windows.items():
    print(name)
    print('  first 5 =', np.round(w[:5], 3))
    print('  last  5 =', np.round(w[-5:], 3))


## 2. ✏️ 你的任务：描述一个窗的形状

**推理路线**：
1. 函数接收一个 numpy 数组 `w`（shape `(N,)`），不依赖窗的类型名，只看数值。
2. 边界值：`w[0]` 取首端，`w[-1]` 取尾端——这两个数直接反映窗对边缘的处理方式。
3. 峰值：`w.max()` 取最大权重；`int(np.argmax(w))` 取峰值所在索引——对对称窗，峰值应在 `N//2 - 1` 附近。
4. 返回四元组 `(first, last, peak, peak_idx)`，让调用方可以一行数字对比不同窗的差异。

**参考输入输出**：`N=10, Hann` → first≈0, last≈0, peak≈1.0, peak_idx=4（中心）；`Rectangular` → first=1, last=1, peak=1, peak_idx=0

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `describe_window` 前明确三件事：
- 输入：`w`（shape `(N,)` 的窗权重数组，值域 0~1）
- 关键步骤：取 `w[0]`、`w[-1]`、`w.max()`、`int(np.argmax(w))`
- 返回：四元组 `(first, last, peak, peak_idx)`，依次是首端值、尾端值、峰值、峰值位置索引

In [ ]:
def describe_window(name, w):
    # ✏️ TODO: 打印 name、长度 len(w)、首尾值、峰值与峰值下标
    print(name, '...')  # ← 改这里，打印有用信息


## 3. 在三种窗上调用 + 自动检查

In [ ]:
N = 64
windows = {'Hann': hann(N), 'Hamming': hamming(N), 'Blackman': blackman(N)}
all_ok = True
for name, w in windows.items():
    assert w.shape[0] == N
    try:
        result = describe_window(name, w)
    except NotImplementedError:
        print('⬜ 未实现')
        all_ok = False
        break
    if result is None:
        print('⬜ 未实现')
        all_ok = False
        break
    first, last, peak, peak_idx = result
    assert abs(first - w[0]) < 1e-9, f'{name}: first 值不对'
    assert abs(last - w[-1]) < 1e-9, f'{name}: last 值不对'
    assert abs(peak - w.max()) < 1e-9, f'{name}: peak 值不对'
    assert peak_idx == int(np.argmax(w)), f'{name}: peak_idx 不对'

if all_ok:
    assert abs(hann(N)[0]) < 1e-9, 'Hann 窗两端应为 0'
    print('\n✅ 窗函数性质核对完毕。')

## 4. 画出三种窗对比

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
for name, w in windows.items():
    plt.plot(w, label=name)
plt.title(f'Window functions (N={N})')
plt.xlabel('sample'); plt.ylabel('weight')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4.1 加窗（windowing）前后：同一段信号，不同边缘

下面保留同一段正弦，只改变窗。先看时域（time domain）边缘，再看频域（frequency domain）里主峰旁边的能量。


In [ ]:
import numpy as np, matplotlib.pyplot as plt

N = 64
n = np.arange(N)
signal = np.sin(2*np.pi*5.5*n/N)  # 非整数周期，边缘会接不上
windows = {
    'rectangular': np.ones(N),
    'hann': np.hanning(N),
    'hamming': np.hamming(N),
}

plt.figure(figsize=(9, 3.2))
for name, w in windows.items():
    plt.plot(signal * w, label=name)
plt.title('same signal × different windows')
plt.legend(); plt.grid(True, alpha=.25); plt.show()

plt.figure(figsize=(9, 3.2))
for name, w in windows.items():
    spectrum = np.abs(np.fft.rfft(signal * w))
    plt.plot(spectrum / spectrum.max(), label=name)
plt.title('normalized spectrum after windowing')
plt.legend(); plt.grid(True, alpha=.25); plt.show()


## 5. 源文件阅读：把概念映射回 Aurora

完成窗函数实验后，再打开这两个文件：

- `src/aurora/audio/io.py`：`sine` / `chirp` / `read_wav` / `write_wav`
- `src/aurora/audio/windows.py`：`hann` / `hamming` / `blackman`

阅读时只对照三件事：函数输入、返回数组、数组长度。


## 🎨 图示：从窗 → STFT 分帧 → 频谱 → mel

把后面 L37-L47 要做的事先看一眼，统一观感。

In [ ]:
from aurora.audio import sine
import aurora.aviz as aviz; aviz.style()
long = sine(220.0, duration=0.05, sample_rate=16000)
aviz.windows_overlay(64);

In [ ]:
aviz.framing(long, frame_len=256, hop=128, n_frames=5);   # STFT 分帧

In [ ]:
aviz.spectrogram(long);                                   # 功率谱图

In [ ]:
aviz.mel_filterbank_plot(20, 256, 16000);                 # mel 滤波器组=矩阵

In [ ]:
aviz.mel_spectrogram_plot(long, 16000, 40);               # mel 频谱图

In [ ]:
N = 16
signal = np.ones(N)
for window_name, window in [('rect', np.ones(N)), ('hann', np.hanning(N)), ('hamming', np.hamming(N))]:
    shaped = signal * window
    edge_energy = shaped[0]**2 + shaped[-1]**2
    print(f'{window_name:7s} | 边缘值=({shaped[0]:.3f}, {shaped[-1]:.3f}) | 边缘能量={edge_energy:.3f}')


## 参数实验：旁瓣高度直接对比

对同一段正弦信号（频率恰好在两个 FFT 频点之间），分别用 Rectangular 和 Hann 加窗后做 FFT，观察 Rectangular 的频谱旁瓣高出主瓣约 13 dB，Hann 的旁瓣低约 31 dB。差距来自边缘处理：Rectangular 的硬截断制造虚假高频，Hann 的平滑过渡大幅压制旁瓣。这就是 STFT 几乎总是用 Hann 窗的原因。

进一步实验：把 `signal = np.sin(2*np.pi*5.5*n/N)` 中的 `5.5` 改成整数（如 `6.0`），矩形窗的旁瓣会消失——频率为整数倍时，FFT 周期边界处信号恰好连续，边缘跳变为零，旁瓣污染也随之消失。

## 本课收束

现在能用 `hann(N)`、`hamming(N)`、`blackman(N)` 生成窗权重，并用 `signal * w` 完成加窗，打印边缘值确认压低效果。这对应 Aurora STFT 流程的第一步：`aurora.audio.windows` 里的三个函数在 `stft` 分帧时被逐帧调用。下一节进入 FFT 实现，今天的窗权重将作为 `xw = x * w` 直接输入离散傅里叶变换。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))
